# 04 - Did we break the model?
**Paper:** Section 4.2 (Table 1). The claim: the orthogonalized model loses refusal but keeps most capability.
Light version: WikiText CE, MMLU subset, and KL on harmless prompts.

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/YOUR_GITHUB_USERNAME/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
try:  # autoreload picks up edits to src/ without restarting
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:  # Colab's IPython autoreload imports `imp`, which Python 3.12+ removed
    pass
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

In [ ]:
import copy, torch.nn.functional as F
from refusal_direction import artifacts
from refusal_direction.orthogonalize import orthogonalize_weights
from refusal_direction.capability import load_wikitext, mean_cross_entropy, load_mmlu, mmlu_accuracy
direction, meta = artifacts.load_direction(cfg)
edited = copy.deepcopy(model); orthogonalize_weights(edited, direction.to(model.cfg.device))

# control: orthogonalize a random direction of the same norm
torch.manual_seed(cfg.seed)
rand_edited = copy.deepcopy(model); orthogonalize_weights(rand_edited, torch.randn_like(direction).to(model.cfg.device))

In [ ]:
wiki = load_wikitext(n=200)
mmlu = load_mmlu(n=200)
S = tok(data["harmless_test"])
base_lp = last_token_logits(model, *S).log_softmax(-1)
rows = {}
for name, m in [("base", model), ("orthogonalized", edited), ("random_orth", rand_edited)]:
    lp = last_token_logits(m, *S).log_softmax(-1)
    rows[name] = {
        "wikitext_CE": mean_cross_entropy(m, wiki),
        "mmlu_acc": mmlu_accuracy(m, mmlu, cfg.system_prompt),
        "harmless_KL_vs_base": F.kl_div(lp, base_lp, log_target=True, reduction="batchmean").item(),
    }
table = pd.DataFrame(rows).T
artifacts.save_json(cfg, "capability.json", rows)
table

## Questions to write up
1. Compare the CE / MMLU deltas to Table 1 for Qwen 1.8B. Bigger or smaller drop?
2. 200 MMLU questions gives a standard error of about 3.5 points. Is your difference significant? (Run a paired bootstrap if you want to be sure.)
3. The paper notes TruthfulQA drops noticeably. Why might removing refusal hurt truthfulness specifically?